# 01 Tensor 基础

PyTorch 环境验证与基础 Tensor 操作

**环境要求**:
- Python 3.10+
- PyTorch 2.0+
- Apple M4 Pro (MPS 加速)

In [1]:
import torch
import time

## 1. 环境检查

检查 PyTorch 版本和 MPS 是否可用

In [2]:
print(f"PyTorch 版本: {torch.__version__}")
print(f"MPS 可用: {torch.backends.mps.is_available()}")
print(f"MPS 已构建: {torch.backends.mps.is_built()}")

# 确定可用设备
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"✓ 将使用 MPS 加速 (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("✗ MPS 不可用，将使用 CPU")

print(f"\n当前设备: {device}")

PyTorch 版本: 2.11.0+cpu
MPS 可用: False
MPS 已构建: False
✗ MPS 不可用，将使用 CPU

当前设备: cpu


## 2. Tensor 基础操作

### 2.1 创建 Tensor

In [3]:
# 从列表创建
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"从列表创建: {x}")
print(f"形状: {x.shape}")
print(f"数据类型: {x.dtype}")

从列表创建: tensor([[1, 2, 3],
        [4, 5, 6]])
形状: torch.Size([2, 3])
数据类型: torch.int64


### 2.2 常用创建方法

In [5]:
# 全零
zeros = torch.zeros(2, 3)
print(f"zeros(2,3):\n{zeros}")

# 全一
ones = torch.ones(2, 3)
print(f"\nones(2,3):\n{ones}")

# 均匀分布随机数 [0, 1)
rand = torch.rand(2, 3)
print(f"\nrand(2,3):\n{rand}")

# 标准正态分布随机数
randn = torch.randn(2, 3)
print(f"\nrandn(2,3):\n{randn}")

zeros(2,3):
tensor([[0., 0., 0.],
        [0., 0., 0.]])

ones(2,3):
tensor([[1., 1., 1.],
        [1., 1., 1.]])

rand(2,3):
tensor([[0.5089, 0.6143, 0.1940],
        [0.7232, 0.9315, 0.4002]])

randn(2,3):
tensor([[ 0.6177,  0.0022, -1.5692],
        [-0.0295, -0.3266, -0.6461]])


### 2.3 Tensor 运算

In [7]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a = {a}")
print(f"b = {b}")
print(f"a + b = {a + b}")
print(f"a * b = {a * b}")
print(f"a @ b (点积) = {a @ b}")

a = tensor([1., 2., 3.])
b = tensor([4., 5., 6.])
a + b = tensor([5., 7., 9.])
a * b = tensor([ 4., 10., 18.])
a @ b (点积) = 32.0


### 2.4 形状操作

In [8]:
x = torch.arange(12)
print(f"原始: {x}, 形状: {x.shape}")

x_reshaped = x.reshape(3, 4)
print(f"\nreshape(3,4):\n{x_reshaped}")

x_view = x.view(2, 6)
print(f"\nview(2,6):\n{x_view}")

原始: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]), 形状: torch.Size([12])

reshape(3,4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

view(2,6):
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])


## 3. 设备操作

在 CPU 和 MPS 之间移动 Tensor

In [9]:
# 在指定设备上创建 Tensor
x = torch.randn(3, 3, device=device)
print(f"直接在 {device} 上创建: x.device = {x.device}")

直接在 cpu 上创建: x.device = cpu


In [10]:
# 设备间移动
cpu_tensor = torch.randn(3, 3)
print(f"CPU tensor: {cpu_tensor.device}")

gpu_tensor = cpu_tensor.to(device)
print(f"移动到 {device}: {gpu_tensor.device}")

back_to_cpu = gpu_tensor.cpu()
print(f"移回 CPU: {back_to_cpu.device}")

CPU tensor: cpu
移动到 cpu: cpu
移回 CPU: cpu


## 4. 自动微分 (Autograd)

PyTorch 的自动微分系统

In [12]:
# 创建需要梯度的 Tensor
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x = {x}")

# 前向计算: y = x² + 2x + 1
y = x ** 2 + 2 * x + 1
print(f"y = x² + 2x + 1 = {y}")

# 计算梯度
z = y.sum()
z.backward()

# dy/dx = 2x + 2
print(f"\ndy/dx = 2x + 2 = {x.grad}")
print(f"验证: 2*{x.data} + 2 = {2*x.data + 2}")

x = tensor([2., 3.], requires_grad=True)
y = x² + 2x + 1 = tensor([ 9., 16.], grad_fn=<AddBackward0>)

dy/dx = 2x + 2 = tensor([6., 8.])
验证: 2*tensor([2., 3.]) + 2 = tensor([6., 8.])


## 5. 性能对比: CPU vs MPS

矩阵乘法性能测试

In [13]:
sizes = [1000, 2000, 4000]

for size in sizes:
    print(f"\n矩阵大小: {size}x{size}")
    print("-" * 30)
    
    # CPU 性能
    a_cpu = torch.randn(size, size)
    b_cpu = torch.randn(size, size)
    
    start = time.perf_counter()
    for _ in range(10):
        c_cpu = torch.mm(a_cpu, b_cpu)
    cpu_time = (time.perf_counter() - start) / 10
    print(f"CPU: {cpu_time*1000:.2f} ms")
    
    # MPS 性能
    if device.type == "mps":
        a_mps = a_cpu.to(device)
        b_mps = b_cpu.to(device)
        
        # 预热
        _ = torch.mm(a_mps, b_mps)
        torch.mps.synchronize()
        
        start = time.perf_counter()
        for _ in range(10):
            c_mps = torch.mm(a_mps, b_mps)
        torch.mps.synchronize()
        mps_time = (time.perf_counter() - start) / 10
        
        speedup = cpu_time / mps_time
        print(f"MPS: {mps_time*1000:.2f} ms")
        print(f"加速比: {speedup:.2f}x")


矩阵大小: 1000x1000
------------------------------
CPU: 7.62 ms

矩阵大小: 2000x2000
------------------------------
CPU: 48.29 ms

矩阵大小: 4000x4000
------------------------------
CPU: 382.41 ms


## 总结

本 notebook 涵盖了:

1. **环境检查** - 验证 PyTorch 和 MPS 可用性
2. **Tensor 基础** - 创建、运算、形状操作
3. **设备操作** - CPU/MPS 间的数据移动
4. **自动微分** - requires_grad 和 backward()
5. **性能对比** - MPS 加速效果

下一步：学习 `nn.Module` 构建神经网络